In [ ]:
import os
import sys
sys.path.insert(0, os.path.expanduser("source"))

from collections import defaultdict
import torch
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import ContrastiveLoss
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

from data_handling import (
    load_data,
    ChunkBatchSampler, 
    custom_collate,
    get_custom_batched_tokenized,
    get_flattened_contrastive_dataset,
    get_hierarchical_contrastive_dataset
)
from evaluation import (
    get_classes_and_embeddings, 
    get_predictions_sbert_binary_classifier,
    evaluate_bcubed, 
    evaluate_ari,
    fuzzy_silhouette_score
)
from utils import avg, to_coarse, logits_to_multihot, multihot_to_list_of_classes, label_set_to_multihot, get_taxonomy_string, get_id_to_narrative_dict
from training import HierarchicalContrastiveLoss
from visualization import get_flat_cmap, get_hierarchical_cmap, plot_results, plot_hierarchical_results

%matplotlib inline

In [ ]:
dataset_name = "CARDS"
dataset_type = {"polynarrative": "multilabel", "CARDS": "multiclass"}.get(dataset_name)

<h2>Load the dataset:</h2>

In [ ]:
exclude_other = False
dataset, (dataset_train, dataset_dev, dataset_test), id_to_class, class_to_id = load_data(dataset_name, exclude_other, False)

<h2>Using SentenceTransformerTrainer:</h2>

In [ ]:
contrastive_type = "hierarchical"
margin = 2/3

# Turn the dataset into a contrastive one
if contrastive_type == "hierarchical":
    contrastive_train = get_hierarchical_contrastive_dataset(dataset["train"].to_pandas(), seed=42, desired_total_count=200000)
    contrastive_dev = get_hierarchical_contrastive_dataset(dataset["dev"].to_pandas(), seed=42, desired_total_count=5000)
elif contrastive_type == "flat":
    contrastive_train = get_flattened_contrastive_dataset(dataset["train"].to_pandas(), seed=42, desired_total_count=200000)
    contrastive_dev = get_flattened_contrastive_dataset(dataset["dev"].to_pandas(), seed=42, desired_total_count=5000)

In [ ]:
model_name = "all-mpnet-base-v2"
batch_size = 64
lr = 5e-05

model = SentenceTransformer(model_name, device="cuda")
if contrastive_type == "hierarchical":
    tensor_mapping = torch.tensor([[0,0],[1,1],[1,0]])
    loss_fn = HierarchicalContrastiveLoss(model, tensor_mapping, margin=margin)
elif contrastive_type == "flat":
    loss_fn = ContrastiveLoss(model)
model_path = f"models/{dataset_name}/contrastive_{model_name}_{contrastive_type}_lr_{lr}_batch_size_{batch_size}"

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir=model_path,
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=16,
    learning_rate=lr,
    warmup_ratio=0.1,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=10000 // batch_size,
    save_strategy="steps",
    save_steps=10000 // batch_size,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True
)

trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=contrastive_train,
    eval_dataset=contrastive_dev,
    args=args,
    loss=loss_fn
)

trainer.train()
trainer.save_model(model_path)

<h3>Load a trained model</h3>

In [ ]:
# Load already trained model instead
model_name = "all-mpnet-base-v2"
batch_size = 64
lr = 5e-05
contrastive_type = "hierarchical"
model_path = f"models/{dataset_name}/contrastive_{model_name}_{contrastive_type}_lr_{lr}_batch_size_{batch_size}"

model = SentenceTransformer(model_path, device="cuda")

<h2>Evaluation</h2>

In [ ]:
dev_batch_size = 1 if dataset_type == "multilabel" else 32
dataloader_dev = torch.utils.data.DataLoader(dataset_dev, batch_size=dev_batch_size)

In [ ]:
aggregation = None
truth, embs = get_classes_and_embeddings(model, dataloader_dev, dataset_type, aggregation=aggregation)

<h2>Multiclass cluster evaluation</h2>

In [ ]:
def get_ari(embs, truth, t):
    algo = AgglomerativeClustering(n_clusters=None, metric='cosine', distance_threshold=t, linkage='complete')
    labels = algo.fit_predict(embs)
    try:
        silhouette = silhouette_score(embs, labels, metric="cosine")
    except ValueError:
        silhouette = 0
    coarse_ari = evaluate_ari(truth, labels, class_to_id, coarse=True)
    ari = evaluate_ari(truth, labels, class_to_id, coarse=False)
    return silhouette, coarse_ari, ari

<p>Compute ARI and silhouette scores for different distance thresholds in agglomerative clustering: </p>

In [ ]:
t_range = np.arange(0, 1.1, 0.1)

silhouette_scores, coarse_aris, aris = zip(*[get_ari(embs, truth, t) for t in t_range]) 

<p>Plot the results: </p>

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(t_range, silhouette_scores, label="Silhouette score")
ax.plot(t_range, coarse_aris, label="ARI (coarse)")
ax.plot(t_range, aris, label="ARI (fine)")
if contrastive_type == "flat":
    ax.vlines(margin, ymin=0, ymax=1, color='black', linestyles='dashed', label="Model margin")
else:
    ax.vlines(margin/2, ymin=0, ymax=1, color='black', linestyles='dashed', label="Model margin(fine)")
    ax.vlines(margin, ymin=0, ymax=1, color='black', linestyles='dotted', label="Model margin (coarse)")
ax.set_xlabel("Distance threshold")
ax.legend(loc='lower left')
ax.set_title(f"{model_name} ({contrastive_type})")

<p>Choose the best clustering parameters for plotting embeddings (further down)</p>

In [ ]:
coarse_d = 0.6
fine_d = 0.4

algo = AgglomerativeClustering(n_clusters=None, metric='cosine', distance_threshold=coarse_d, linkage='complete')
coarse_preds = algo.fit_predict(embs)
algo = AgglomerativeClustering(n_clusters=None, metric='cosine', distance_threshold=fine_d, linkage='complete')
fine_preds = algo.fit_predict(embs)

<p>Visualize the embeddings and their true and predicted labels: </p>

In [ ]:
tSNE = TSNE(init="random", learning_rate="auto", metric="cosine")
embs_2d = tSNE.fit_transform(embs)
plot_hierarchical_results(truth, coarse_preds, fine_preds, embs_2d, id_to_class, loc="lower right")

<h2>Multi-label cluster evaluation</h2>

<h4>Agglomerative clustering</h4>
<p>Use agglomerative clustering on each chunk (requires having set aggregation=None when getting embeddings): </p>

In [ ]:
coarse_f1s = defaultdict()
fine_f1s = defaultdict()
fuzzy_silhouettes = defaultdict()

d_range = np.arange(0.1, 1.1, 0.1)

best_silhouette = -1
best_silhouette_n = None

truth_ids = [{class_to_id[c] for c in multihot_to_list_of_classes(t)} for t in truth]
lengths = [len(e) for e in embs]
concat_embs = np.concatenate(embs)

best_coarse = 0
best_fine = 0
best_coarse_d = None
best_fine_d = None

for d in d_range:
    algo = AgglomerativeClustering(n_clusters=None, metric='cosine', distance_threshold=d, linkage='complete')
    chunk_labels = algo.fit_predict(concat_embs)
    preds = np.array_split(chunk_labels, np.cumsum(lengths[:-1]))
    preds = [set(p) for p in preds]
    pred_ids = preds
    
    try:
        fuzzy_silhouettes[n] = fuzzy_silhouette_score(embs, fuzzy_labels)
    except ValueError:
        fuzzy_silhouettes[n] = -1
    if fuzzy_silhouettes[n] > best_silhouette:
        best_silhouette = fuzzy_silhouettes[n]
        best_silhouette_n = n
        
    coarse_p, coarse_r, coarse_f1 = evaluate_bcubed(truth_ids, pred_ids, coarse=True)
    fine_p, fine_r, fine_f1 = evaluate_bcubed(truth_ids, pred_ids, coarse=False)
    fine_f1s[d] = fine_f1
    coarse_f1s[d] = coarse_f1
    if fine_f1 > best_fine:
        best_fine = fine_f1
        best_fine_d = d
    if coarse_f1 > best_coarse:
        best_coarse = coarse_f1
        best_coarse_d = d

print(f"Max fuzzy silhouette score was {best_silhouette} (n={best_silhouette_n})")
print(f"Max coarse and fine F1 were {best_coarse} (t={best_coarse_d}), {best_fine} (t={best_fine_d})")

<p>Choose the best parameters and do a prediction for plotting: </p>

In [ ]:
d = 0.6
algo = AgglomerativeClustering(n_clusters=None, metric='cosine', distance_threshold=d, linkage='complete')
chunk_labels = algo.fit_predict(concat_embs)
preds = np.array_split(chunk_labels, np.cumsum(lengths[:-1]))
preds = [set(p) for p in preds]

<h4>Possibilistic c-means</h4>
<p>As an alternative, use possibilistic c-means on average embeddings (requires having cloned the repo at https://github.com/holtskinner/PossibilisticCMeans): </p>

In [6]:
# To get `pcm`, clone the repository at https://github.com/holtskinner/PossibilisticCMeans
def get_fuzzy_labels(embs, n_clusters):
    _, _, u, _, _, _ = pcm(embs.transpose(), n_clusters, 2, 0.0005, 100000, metric="cosine")
    return u.transpose()

# Compute fuzzy c means or possibilistic c means on average embeddings
def get_hard_labels(fuzzy_labels, t):
    return [set(np.where(l)[0].tolist()) or {np.argmax(fuzzy_labels[i])} for i, l in enumerate(fuzzy_labels > t)]

In [ ]:
coarse_f1s = defaultdict()
fine_f1s = defaultdict()
fuzzy_silhouettes = defaultdict()

# n is the amount of clusters, t the prediction threshold
n_range = np.arange(2, len(id_to_class), 1)
t_range = np.arange(0.1, 1.1, 0.05)

best_silhouette = -1
best_silhouette_n = None

best_fine = 0
best_coarse = 0
best_fine_t = None
best_coarse_t = None
best_fine_n = None
best_coarse_n = None

truth_ids = [{class_to_id[c] for c in multihot_to_list_of_classes(t)} for t in truth]

for n in n_range:
    fuzzy_labels = get_fuzzy_labels(embs, n)
    fine_f1s[n] = defaultdict()
    coarse_f1s[n] = defaultdict()
    try:
        fuzzy_silhouettes[n] = fuzzy_silhouette_score(embs, fuzzy_labels)
    except ValueError:
        fuzzy_silhouettes[n] = -1
    if fuzzy_silhouettes[n] > best_silhouette:
        best_silhouette = fuzzy_silhouettes[n]
        best_silhouette_n = n

    for t in t_range:
        pred_ids = get_hard_labels(fuzzy_labels, t)
        _, _, coarse_f1 = evaluate_bcubed(truth_ids, pred_ids, coarse=True)
        _, _, fine_f1 = evaluate_bcubed(truth_ids, pred_ids, coarse=False)
        fine_f1s[n][t] = fine_f1
        coarse_f1s[n][t] = coarse_f1
        if fine_f1 > best_fine:
            best_fine = fine_f1
            best_fine_n = n
            best_fine_t = t
        if coarse_f1 > best_coarse:
            best_coarse = coarse_f1
            best_coarse_n = n
            best_coarse_t = t


print(f"Max fuzzy silhouette score was {best_silhouette} (n={best_silhouette_n})")
print(f"Best coarse and fine F1 were {best_coarse} (n={best_coarse_n}, t={best_coarse_t}), {best_fine} (n={best_fine_n},t={best_fine_t})")

<p>Choose the best parameters and do a prediction for plotting: </p>

In [ ]:
# Do just one prediction to inspect the embeddings
n, t = (42, 0.85)
fuzzy_labels = get_fuzzy_labels(embs, n)
cluster_labels = get_hard_labels(fuzzy_labels, t)

<p>Visualize embeddings along with predicted and ground truth labels: </p>

In [ ]:
tSNE = TSNE(init="random", learning_rate="auto", metric="cosine")
if aggregation == "mean":
    embs_2d = tSNE.fit_transform(embs)
else:
    avg_embs = np.vstack([e.mean(axis=0) for e in embs])
    embs_2d = tSNE.fit_transform(avg_embs)
plot_results(truth, preds, embs_2d, id_to_class, dataset_type)